# Lab 4 — Naïve Bayes Classifier
**Dataset:** `creditcard.csv`  ·  `Class`: 0 = Genuine, 1 = Fraud  — **do not change**

**Objective**
- **Part 1:** implement **Gaussian Naive Bayes from scratch** (NumPy / Pandas only)
- **Part 2:** reproduce with **scikit-learn**, then compare on the same split

**Rules**
- Section 0 scaffolding is **GIVEN** — do not modify.
- Complete every `# TODO:` and replace each `___`.
- Both parts reuse the same `X_train / X_test / y_train / y_test` and the same `evaluate()` helper.

## Section 0 — Setup & Shared Pipeline  *(GIVEN — do not modify)*

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, recall_score, precision_score,
                             accuracy_score, f1_score, roc_auc_score)

In [2]:
df = pd.read_csv("creditcard.csv")
print("shape:", df.shape)
print(df["Class"].value_counts())      # class imbalance: fraud is rare

shape: (284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


In [3]:
# Basic preprocessing (GIVEN)
df = df.drop(["Time"], axis=1)                                                 # drop non-informative column
df["scaled_Amount"] = StandardScaler().fit_transform(df["Amount"].values.reshape(-1, 1))
df = df.drop(["Amount"], axis=1)

In [4]:
y = df["Class"].values                        # y: [N]
X = df.drop(["Class"], axis=1).values         # X: [N, D]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("train:", X_train.shape, "| test:", X_test.shape, "| fraud in test:", int(y_test.sum()))

train: (227845, 29) | test: (56962, 29) | fraud in test: 98


In [5]:
def evaluate(y_true, y_pred, y_prob=None, title=""):
    """Shared evaluation printer (GIVEN)."""
    print(f"=== {title} ===")
    print("confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print("recall   :", round(recall_score(y_true, y_pred), 4))
    print("precision:", round(precision_score(y_true, y_pred), 4))
    print("f1       :", round(f1_score(y_true, y_pred), 4))
    print("accuracy :", round(accuracy_score(y_true, y_pred), 4))
    if y_prob is not None:
        print("ROC AUC  :", round(roc_auc_score(y_true, y_prob), 4))

## Part 1 — Implement from scratch (NumPy / Pandas only)

Gaussian NB pipeline:
1. `fit` \u2192 priors $P(c)$, per-class per-feature `mean` $\mu_{c,d}$, `var` $\sigma^2_{c,d}$
2. `_log_gaussian` \u2192 $\log \mathcal{N}(x;\mu,\sigma^2)$
3. `_log_posterior` \u2192 $\log P(c) + \sum_d \log P(x_d \mid c)$
4. `predict` \u2192 MAP rule: $\arg\max_c$ log posterior

In [6]:
class GaussianNBScratch:
    """Gaussian Naive Bayes — student implementation."""

    def fit(self, X, y):
        # X: [N, D], y: [N]
        self.classes_ = np.unique(y)
        self.priors_, self.mean_, self.var_ = {}, {}, {}
        n_samples = X.shape[0]
        for c in self.classes_:
            Xc = X[y == c]                       # Xc: [N_c, D]
            # TODO: prior  P(c) = N_c / N
            self.priors_[c] = Xc.shape[0] / n_samples
            # TODO: per-feature mean of class c        -> shape [D]
            self.mean_[c] = Xc.mean(axis=0)
            # TODO: per-feature variance of class c     -> shape [D]
            self.var_[c] = Xc.var(axis=0)
        return self

    def _log_gaussian(self, x, mean, var):
        # x, mean, var: [D]  ->  [D] : element-wise log N(x; mean, var)
        eps = 1e-9
        # TODO: log Gaussian pdf
        #   = -0.5*log(2*pi*(var+eps)) - (x - mean)**2 / (2*(var+eps))
        return -0.5 * np.log(2 * np.pi * (var + eps)) - (x - mean) ** 2 / (2 * (var + eps))

    def _log_posterior(self, x):
        # x: [D]  ->  dict {class: unnormalized log posterior}
        log_post = {}
        for c in self.classes_:
            # TODO: log_prior = log P(c)
            log_prior = np.log(self.priors_[c])
            # TODO: log_likelihood = sum over D features of _log_gaussian(...)
            log_likelihood = np.sum(self._log_gaussian(x, self.mean_[c], self.var_[c]))
            log_post[c] = log_prior + log_likelihood
        return log_post

    def predict(self, X):
        # X: [N, D]  ->  y_pred: [N]
        preds = []
        for x in X:                              # x: [D]
            log_post = self._log_posterior(x)
            # TODO: MAP rule — pick the class with the largest log posterior
            best_c = max(log_post, key=log_post.get)
            preds.append(best_c)
        return np.array(preds)

In [7]:
model = GaussianNBScratch()
model.fit(X_train, y_train)            # TODO: fit model on the training set
y_pred_scratch = model.predict(X_test) # TODO: predict class labels on X_test
evaluate(y_test, y_pred_scratch, None, "Part 1 - GaussianNB (from scratch)")

=== Part 1 - GaussianNB (from scratch) ===
confusion matrix:
 [[55534  1330]
 [   15    83]]
recall   : 0.8469
precision: 0.0587
f1       : 0.1099
accuracy : 0.9764


## Part 2 — Using scikit-learn

Reproduce the classifier with `sklearn.naive_bayes.GaussianNB` on the **same split**.

In [8]:
from sklearn.naive_bayes import GaussianNB

clf = GaussianNB()                        # TODO: instantiate GaussianNB
clf.fit(X_train, y_train)                 # TODO: fit clf on (X_train, y_train)
y_pred_sk = clf.predict(X_test)           # TODO: predict class labels on X_test
y_prob_sk = clf.predict_proba(X_test)[:, 1]  # TODO: predict P(class=1) on X_test  (predict_proba, column 1)
evaluate(y_test, y_pred_sk, y_prob_sk, "Part 2 - sklearn GaussianNB")

=== Part 2 - sklearn GaussianNB ===
confusion matrix:
 [[55534  1330]
 [   15    83]]
recall   : 0.8469
precision: 0.0587
f1       : 0.1099
accuracy : 0.9764
ROC AUC  : 0.9629


## Section 3 — Compare

| Metric | Part 1 (scratch) | Part 2 (sklearn) |
|---|---|---|
| Recall | 0.8469 | 0.8469 |
| Precision | 0.0587 | 0.0587 |
| F1 | 0.1099 | 0.1099 |
| Accuracy | 0.9764 | 0.9764 |

**Q:** Accuracy rất cao nhưng vì sao **recall** mới là metric quan trọng trên dataset này? →

Vì dữ liệu bị mất cân bằng nặng: giao dịch gian lận chỉ chiếm một phần rất nhỏ so với giao dịch bình thường. Khi đó một mô hình chỉ cần đoán mọi giao dịch là bình thường cũng đã có accuracy rất cao, dù nó không phát hiện được ca gian lận nào. Accuracy cao vì thế không nói lên mô hình có thực sự bắt được gian lận hay không.

Recall thì đo đúng cái ta cần: trong số giao dịch gian lận thật, mô hình phát hiện được bao nhiêu. Bỏ lọt một ca gian lận gây thiệt hại thật, còn báo nhầm một giao dịch bình thường chỉ tốn công kiểm tra lại. Vì hai loại sai có cái giá rất khác nhau nên recall mới là thước đo cần quan tâm, và có thể chấp nhận precision thấp hơn để đổi lấy recall cao.